# Treinamento inicial YOLO para ovitrampas

Este notebook usa os cinco pares de imagens originais/anotadas já preparados. Traços vermelhos isolados são convertidos em candidatos a `ovo_individual`; traços conectados ou grandes viram `aglomerado`, que fica separado da contagem e exige validação.

**Limite deste primeiro treino:** há somente cinco imagens anotadas (quatro para treino e uma para validação). O modelo é um baseline de exploração; previsões nas demais fotos são sugestões, não contagens finais. As regiões red-mark detectadas automaticamente e as caixas aproximadas devem ser revisadas antes de interpretar métricas.


In [4]:
pip install ultralytics opencv-python pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path
import cv2
import shutil
import random

ROOT = Path.cwd()
IMAGES = ROOT / "data" / "images"
ANNOTATED = ROOT / "data" / "annotated"
DATASET = ROOT / "data" / "yolo_dataset_revisado"

# As imagens anotadas foram salvas como image1.png ... image5.png,
# na mesma ordem das cinco fotos amostradas.
PAIRING = {
    "image1.png": "2-CE-1.JPEG",
    "image2.png": "2-CE-15.JPEG",
    "image3.png": "2-NE-1.JPEG",
    "image4.png": "2-NE-10.JPEG",
    "image5.png": "2-NE-15.JPEG",
    "image7.png": "8-CE-10.JPEG",
}
CLASS_NAMES = ["ovo_individual", "aglomerado"]

# Extrai os traços vermelhos desenhados pelo revisor. Marcas próximas ou
# conectadas viram uma região da classe aglomerado e devem ser validadas.
def components(mask, min_area=80):
    count, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, 8)
    found = []
    for i in range(1, count):
        x, y, w, h, area = map(int, stats[i])
        if area >= min_area:
            found.append((x, y, w, h, area, centroids[i]))
    return found

def extract_regions(marked_path):
    image = cv2.imread(str(marked_path))
    if image is None:
        raise FileNotFoundError(marked_path)
    b, g, r = cv2.split(image)
    # Vermelho = ovo individual; não agrupamos por proximidade ou tamanho.
    red = ((r > 130) & (g < 120) & (b < 120)).astype("uint8") * 255
    # Laranja = aglomerado confirmado pelo revisor.
    orange = ((r > 170) & (g > 55) & (g < 175) & (b < 100)).astype("uint8") * 255
    # Amarelo = candidato incerto; fica fora do treino e vai para revisão.
    yellow = ((r > 170) & (g > 190) & (b < 120)).astype("uint8") * 255
    regions = []
    for x, y, w, h, area, center in components(red):
        cx, cy = center
        bw, bh = 16, 22
        x1, y1 = max(0, int(cx-bw/2)), max(0, int(cy-bh/2))
        x2, y2 = min(image.shape[1], x1+bw), min(image.shape[0], y1+bh)
        regions.append((0, x1, y1, x2, y2))
    for x, y, w, h, area, center in components(orange, min_area=120):
        regions.append((1, x, y, x+w, y+h))
    uncertain = components(yellow, min_area=120)
    return image.shape[1], image.shape[0], regions, len(uncertain)
# Recria somente a pasta de preparo, sem mexer nas fotos originais/anotações.
if DATASET.exists():
    raise FileExistsError(f"A pasta {DATASET} já existe; mova-a ou renomeie-a antes de recriar o conjunto.")
for split in ("train", "val"):
    (DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)

items = list(PAIRING.items())
random.Random(42).shuffle(items)
val_names = {name for name, _ in items[:1]}
summary = []
for marked_name, source_name in PAIRING.items():
    source = IMAGES / source_name
    marked = ANNOTATED / marked_name
    if not source.exists() or not marked.exists():
        raise FileNotFoundError(f"Confira foto original/anotada: {source} / {marked}")
    width, height, regions, uncertain_count = extract_regions(marked)
    split = "val" if marked_name in val_names else "train"
    image_out = DATASET / "images" / split / source.name
    label_out = DATASET / "labels" / split / f"{source.stem}.txt"
    shutil.copy2(source, image_out)
    lines = []
    for cls, x1, y1, x2, y2 in regions:
        xc, yc = ((x1+x2)/2)/width, ((y1+y2)/2)/height
        bw, bh = (x2-x1)/width, (y2-y1)/height
        lines.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    label_out.write_text("\n".join(lines), encoding="utf-8")
    summary.append({"imagem": source_name, "split": split,
                    "ovos_individuais": sum(r[0] == 0 for r in regions),
                    "aglomerados_confirmados": sum(r[0] == 1 for r in regions),
                    "candidatos_amarelos_para_revisao": uncertain_count})

config_text = f"""path: {DATASET.resolve().as_posix()}
train: images/train
val: images/val
names:
  0: ovo_individual
  1: aglomerado
"""
(DATASET / "data.yaml").write_text(config_text, encoding="utf-8")
summary


In [ ]:
from ultralytics import YOLO

# Modelo leve pré-treinado; use GPU quando disponível.
model = YOLO("yolo26n.pt")
results = model.train(
    data=str(DATASET / "data.yaml"),
    epochs=80,
    imgsz=1280,
    batch=2,
    patience=20,
    project=str(ROOT / "runs" / "ovitrampas"),
    name="baseline_revisado",
    exist_ok=False,
)


In [ ]:
from pathlib import Path
import random
import pandas as pd
from ultralytics import YOLO

ROOT = Path.cwd()
IMAGES = ROOT / "data" / "images"
PAIRING = {
    "image1.png": "2-CE-1.JPEG",
    "image2.png": "2-CE-15.JPEG",
    "image3.png": "2-NE-1.JPEG",
    "image4.png": "2-NE-10.JPEG",
    "image5.png": "2-NE-15.JPEG",
    "image7.png": "8-CE-10.JPEG",
}
best = ROOT / "runs" / "ovitrampas" / "baseline_revisado" / "weights" / "best.pt"
model = YOLO(str(best))
seed_images = set(PAIRING.values())
N_IMAGENS_TESTE = 30  # Altere este valor para testar mais ou menos imagens.
files = sorted(p for p in IMAGES.iterdir() if p.suffix.lower() in {".jpg", ".jpeg"} and p.name not in seed_images)
rng = random.Random(42)
files = rng.sample(files, min(N_IMAGENS_TESTE, len(files)))
print(f"Testando {len(files)} imagens nesta rodada.")
rows = []
for path in files:
    result = model.predict(source=str(path), imgsz=1280, conf=0.20, verbose=False)[0]
    names = result.names
    individual = sum(1 for c in result.boxes.cls.tolist() if names[int(c)] == "ovo_individual")
    clusters = sum(1 for c in result.boxes.cls.tolist() if names[int(c)] == "aglomerado")
    rows.append({"imagem": path.name, "ovos_individuais_detectados": individual,
                 "aglomerados_para_validar": clusters,
                 "aviso": "Revisar aglomerados e detecções; baseline treinado em apenas 5 imagens." if clusters else "Revisar contagem; baseline treinado em apenas 5 imagens."})
output = ROOT / "data" / "contagens_preliminares_yolo_revisado.csv"
pd.DataFrame(rows).to_csv(output, index=False, encoding="utf-8-sig")
print(f"CSV salvo em: {output}")
